### Importera verktyg, läsa in fil, konvertera till CSV

In [ ]:
#importerar in verktyg och bibliotek
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


#läser in vår fil
df_flanker = pd.read_excel("Raw_Flanker.xlsx")

#spara som csv i samma directory
df_flanker.to_csv("Raw_Flanker.csv", index=False)

### Utforska och få en överblick över datan
Titta efter NaN, se vilka kolumner vi behöver och hur vi på andra sätt kan förenkla vår data till vårt syfte

In [ ]:
#utforska datan

df_flanker = pd.read_csv("Raw_Flanker.csv")

print("Flanker info")
df_flanker.info()
display(df_flanker.head())


In [ ]:
#check for NaN

print("Tot NaN for Flanker-test")
df_flanker.isna().sum()



### Cleana vår data
Välja kolumner som vi vill ha kvar, sortera, skapa ny kolumn 'session', ta bort practiceruns, göra 'date' tydligare etc

In [ ]:
# läser in csv filen
df_flanker = pd.read_csv("Raw_Flanker.csv")

In [ ]:
# gör datumet mer lättläst
df_flanker['date'] = pd.to_datetime(df_flanker['date'].astype(str).str.zfill(6), format='%m%d%y', errors='coerce').dt.strftime('%Y-%m-%d')

In [ ]:
# sortera kronologiskt efter deltagare, datum och tid så sessionerna hamnar i rätt ordning
df_flanker = df_flanker.sort_values(by=['subjectid', 'date', 'time'])

In [ ]:
# skapa 'session'-kolumnen per deltagare (1 = tidigaste tillfället, 2 = nästa, 3 = sista)
df_flanker['session'] = df_flanker.groupby('subjectid')['date'].transform(lambda x: pd.factorize(x)[0] + 1)

In [ ]:
# ta bort övningsförsök
df_clean = df_flanker[df_flanker['values.practice'] == 0].copy()

In [ ]:
# ta bort session 2 och 3
df_clean = df_clean[~df_clean['session'].isin([2, 3])].copy()

In [ ]:
# välj enbart de kolumner vi vill behålla (inklusive 'session')
kolumner_att_behålla = ['date', 'time', 'subjectid', 'session', 'correct', 'values.congruence', 'latency']

In [ ]:
# kontrollera att kolumnerna finns
df_clean = df_clean[[col for col in kolumner_att_behålla if col in df_clean.columns]]

## Ta bort outliers

In [ ]:
# Exkludera personer med låg andel rätt 
ACC_GRANS = 0.80  # 80 %

acc_flanker = df_clean.groupby("subjectid")["correct"].mean()

låg = acc_flanker[acc_flanker < ACC_GRANS].index  # deltagare UNDER 80 % i Flanker

print(f"Deltagare före: {df_clean['subjectid'].nunique()}")
print(f"Exkluderas ({len(låg)} st):", list(låg))

df_clean = df_clean[~df_clean["subjectid"].isin(låg)].copy()  # behåller deltagare med ≥ 80 %

print(f"Deltagare efter: {df_clean['subjectid'].nunique()}")

In [ ]:
# ta bort outliers reaktionstid
#spara antalet rader innan vi tar bort gränsvärden
antal_fore = len(df_clean)

#sätta gränsvärden för outliers (200ms nedre gräns, 2000ms övre gräns)
df_clean = df_clean[(df_clean['latency'] >= 200) & (df_clean['latency'] <= 3000)].copy()

#ta reda på hur många vi tagit bort genom att jämföra före och efter
antal_efter = len(df_clean)
antal_borttagna = antal_fore - antal_efter

print(f"Antal borttagna försök: {antal_borttagna}")


### Kontrollera våran data
Se över att allt ser korrekt ut och som vi vill ha det, kan vi göra ytterligare förbättringar.

In [ ]:
#kontrollera våran data efter vi cleanat den

display(df_clean.head())
df_clean.info()

df_clean['subjectid'].nunique()

antal_unika = df_clean['subjectid'].nunique()
print(f"Antal unika deltagare: {antal_unika}")



In [ ]:
# spara ner allt till EN gemensam CSV-fil (ta bort # för att spara ner en ny csv fil med cleanad data)

#df_clean.to_csv("flanker_all_sessions_combined.csv", index=False)

print("Ta bort # ovan för att spara ner en ny 'cleanad' csv, 'flanker_all_sessions_combined.csv'.")

## Importera verktyg, läs in fil, konvertera till csv

In [18]:
#importera verktyg och bibliotek
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


#Läser in ecxel filen

df_stroop = pd.read_excel("Raw_Stroop.xlsx")

#Spara som csv i samma directory

df_stroop.to_csv("Raw_Stroop.csv", index=False)


### Utforska och få en överblick över datan
titta efter NaN, se vilka kolummner vi behöver och hur vi på andra sätt kan förenkla vår data för vårt syfte

In [2]:
#Visa sidan "Variables description" i excel filen

desc_stroop = pd.read_excel("Raw_Stroop.xlsx", sheet_name = "Variables Description")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
desc_stroop

,Variable name,Variable Description
0,build,The Inquisit version used to run the task
1,computer.platform,The operational system the task was run on (win/mac/ios/android)
2,date,The date the task was executed
3,time,The time the task was executed
4,blockcode,"The name of the current block (""Stroop"" for all trials)"
5,blocknum,The number of the current block (2 for all trials)
6,trialcode,The name of the current trial:\n“blackcongruent”=congruent trial with black font\n“blackincongruent”=incongruent trial with black font\n“bluecongruent”=congruent trial with blue font\n“blueincongruent”=incongruent trial with blue font\n“greencongruent”=congruent trial with green font\n“greenincongruent”=incongruent trial with green font\n“redcongruent”=congruent trial with red font\n“redincongruent”=incongruent trial with red font\n“yellowcongruent”=congruent trial with yellow font\n“yellowincongruent”=incongruent trial with yellow font
7,trialnum,The sequence number of the currently recorded trial
8,values.congruency,Provides information about the congruency of the current trial:
9,NaN,1 = congruent (color of word and meaning of word are the same);


In [3]:
#explore the data

print("Stroop info")
df_stroop.info()
display(df_stroop.head())

Stroop info
<class 'pandas.DataFrame'>
RangeIndex: 111300 entries, 0 to 111299
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   build              111300 non-null  str  
 1   computer.platform  111300 non-null  str  
 2   date               111300 non-null  int64
 3   time               111300 non-null  str  
 4   blockcode          111300 non-null  str  
 5   blocknum           111300 non-null  int64
 6   trialcode          111300 non-null  str  
 7   trialnum           111300 non-null  int64
 8   values.congruency  111300 non-null  int64
 9   stimulusitem1      111300 non-null  str  
 10  response           111300 non-null  int64
 11  correct            111300 non-null  int64
 12  latency            111300 non-null  int64
 13  subjectid          111300 non-null  int64
dtypes: int64(8), str(6)
memory usage: 16.7 MB


,build,computer.platform,date,time,blockcode,blocknum,trialcode,trialnum,values.congruency,stimulusitem1,response,correct,latency,subjectid
0,5.0.6.0,win,111917,11:17:22,stroop,2,blackcongruent,2,1,czarny,37,1,1443,1
1,5.0.6.0,win,111917,11:17:22,stroop,2,redincongruent,3,2,zielony,33,0,1268,1
2,5.0.6.0,win,111917,11:17:22,stroop,2,greencongruent,4,1,zielony,33,1,1430,1
3,5.0.6.0,win,111917,11:17:22,stroop,2,redincongruent,5,2,niebieski,36,0,1098,1
4,5.0.6.0,win,111917,11:17:22,stroop,2,greencongruent,6,1,zielony,33,1,2061,1


In [4]:
#check for NaN

print("Tot NaN for Stroop-test")
df_stroop.isna().sum()

Tot NaN for Stroop-test


build                0
computer.platform    0
date                 0
time                 0
blockcode            0
blocknum             0
trialcode            0
trialnum             0
values.congruency    0
stimulusitem1        0
response             0
correct              0
latency              0
subjectid            0
dtype: int64

### Cleana vår data

In [ ]:
#Läs in csv filen
df_stroop = pd.read_csv("Raw_Stroop.csv")

In [5]:
# gör datumet mer lättläst
df_stroop['date'] = pd.to_datetime(df_stroop['date'].astype(str).str.zfill(6), format='%m%d%y', errors='coerce').dt.strftime('%Y-%m-%d')

In [6]:
# sortera kronologiskt efter deltagare, datum och tid så sessionerna hamnar i rätt ordning
df_stroop = df_stroop.sort_values(by=['subjectid', 'date', 'time'])

In [7]:
# skapa 'session'-kolumnen per deltagare (1 = tidigaste tillfället, 2 = nästa, 3 = sista)
df_stroop['session'] = df_stroop.groupby('subjectid')['date'].transform(lambda x: pd.factorize(x)[0] + 1)

In [8]:
# välj enbart de kolumner vi vill behålla (inklusive 'session')
kolumner_att_behålla = ['date', 'time', 'subjectid', 'session', 'correct', 'latency', 'values.congruency']

In [ ]:
# ta bort session 2 och 3
df_clean = df_clean[~df_clean['session'].isin([2, 3])].copy()

In [9]:
# kontrollera att kolumnerna finns
df_clean_stroop = df_stroop[kolumner_att_behålla].copy()

In [10]:
df_clean_stroop.head()

,date,time,subjectid,session,correct,latency,values.congruency
0,2017-11-19,11:17:22,1,1,1,1443,1
1,2017-11-19,11:17:22,1,1,0,1268,2
2,2017-11-19,11:17:22,1,1,1,1430,1
3,2017-11-19,11:17:22,1,1,0,1098,2
4,2017-11-19,11:17:22,1,1,1,2061,1


### Ta bort outliers

In [11]:
#antal unika deltagare
antal_unika = df_clean_stroop['subjectid'].nunique()
print(f"Antal unika deltagare: {antal_unika}")

Antal unika deltagare: 466


In [13]:
#Exkludera personer med låg andel rätt
ACC_GRANS = 0.80  # 80 %

acc_stroop = df_clean_stroop.groupby("subjectid")["correct"].mean()

låg = acc_stroop[acc_stroop < ACC_GRANS].index  # deltagare UNDER 80 % i Stroop

print(f"Deltagare före: {df_clean_stroop['subjectid'].nunique()}")
print(f"Exkluderas ({len(låg)} st):", list(låg))

df_clean_stroop = df_clean_stroop[~df_clean_stroop["subjectid"].isin(låg)].copy()  # behåller deltagare med ≥ 80 %

print(f"Deltagare efter: {df_clean_stroop['subjectid'].nunique()}")

Deltagare före: 466
Exkluderas (4 st): [6, 84, 168, 239]
Deltagare efter: 462


In [14]:
# ta bort outliers reaktionstid
#spara antalet rader innan vi tar bort gränsvärden
antal_fore = len(df_clean_stroop)

#sätta gränsvärden för outliers (200ms nedre gräns, 2000ms övre gräns)
df_clean_stroop = df_clean_stroop[(df_clean_stroop['latency'] >= 200) & (df_clean_stroop['latency'] <= 3000)].copy()

#ta reda på hur många vi tagit bort genom att jämföra före och efter
antal_efter = len(df_clean_stroop)
antal_borttagna = antal_fore - antal_efter

print(f"Antal borttagna försök: {antal_borttagna}")

Antal borttagna försök: 719


### Kontrollera vår data
Se över att allt ser korrekt ut och som vi vill ha det, kan vi göra ytterligare förbättringar

In [15]:
#kontrollera våran data efter vi cleanat den

display(df_clean_stroop.head())
df_clean_stroop.info()

df_clean_stroop['subjectid'].nunique()

antal_unika = df_clean_stroop['subjectid'].nunique()
print(f"Antal unika deltagare: {antal_unika}")

,date,time,subjectid,session,correct,latency,values.congruency
0,2017-11-19,11:17:22,1,1,1,1443,1
1,2017-11-19,11:17:22,1,1,0,1268,2
2,2017-11-19,11:17:22,1,1,1,1430,1
3,2017-11-19,11:17:22,1,1,0,1098,2
4,2017-11-19,11:17:22,1,1,1,2061,1


<class 'pandas.DataFrame'>
Index: 110021 entries, 0 to 111299
Data columns (total 7 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   date               110021 non-null  str  
 1   time               110021 non-null  str  
 2   subjectid          110021 non-null  int64
 3   session            110021 non-null  int64
 4   correct            110021 non-null  int64
 5   latency            110021 non-null  int64
 6   values.congruency  110021 non-null  int64
dtypes: int64(5), str(2)
memory usage: 8.6 MB
Antal unika deltagare: 462
